# ReadPDFData Policy Graph Transformation

## Plan

1. Read `corpus.yaml` for document metadata, extraction fields, aliases, and rules.
2. Convert the PDF, HTML, and text documents in `pdfnhtmls` using Docling while preserving page and section provenance.
3. Use Azure OpenAI structured output to extract payer identity and policy criteria.
4. Verify every extracted criterion against an exact quotation from the source document.
5. Use `ground_truth.yaml` as the reviewed acceptance and reconciliation selector; every selected fact must still resolve to PDF/HTML evidence and ground truth is never inserted as graph source data.
6. Deterministically create a single policy graph containing documents, payers, criteria, sections, and evidence.
7. Block database upload if required validation fails.
8. Upload the validated graph to Apache AGE as `payer_policy_knowledge_graph`.
9. Verify node counts, relationships, citations, and representative policy queries.

The source policy facts always come from `pdfnhtmls`. The corpus YAML is configuration and metadata; reviewed ground truth selects and validates the correct source-backed interpretation.

In [ ]:
# Run only if the notebook environment is missing dependencies.
%pip install "docling>=2.120,<3" "openai>=2.54,<3" "pydantic>=2,<3" pyyaml "psycopg[binary]" python-dotenv

In [1]:
import hashlib
import json
import os
import re
import unicodedata
from pathlib import Path
from typing import Any

import psycopg
import yaml
from dotenv import load_dotenv
from openai import AzureOpenAI
from pydantic import BaseModel, ConfigDict, Field

load_dotenv(override=True)

DATA_ROOT = Path("ReadPDFData")
SOURCE_DIR = DATA_ROOT / "pdfnhtmls"
CORPUS_PATH = DATA_ROOT / "corpus.yaml"
GROUND_TRUTH_PATH = DATA_ROOT / "ground_truth.yaml"
OUTPUT_DIR = DATA_ROOT / "outputs"
CONVERSION_CACHE_DIR = OUTPUT_DIR / "docling"
EXTRACTION_CACHE_DIR = OUTPUT_DIR / "extractions"
VALIDATION_REPORT_PATH = OUTPUT_DIR / "validation_report.json"

GRAPH_NAME = "payer_policy_knowledge_graph"
MODELS = ["gpt-5.6-luna", "gpt-5.4-mini"]
VERIFICATION_MODEL = MODELS[0]
EXTRACTION_MODEL = MODELS[1]

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

client = AzureOpenAI(
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DATABASE = os.getenv("PG_DATABASE")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

PROMPT_VERSION = "policy-extraction-v2-field-checklist"
CONVERSION_VERSION = "docling-standard-ocr-v1"

In [2]:
def load_yaml(path: Path) -> dict[str, Any]:
    data = yaml.safe_load(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"{path} must contain a YAML mapping")
    return data


def load_policy_inputs():
    corpus = load_yaml(CORPUS_PATH)
    ground_truth = load_yaml(GROUND_TRUTH_PATH)

    sources = corpus.get("sources") or []
    if not isinstance(sources, list) or not sources:
        raise ValueError("corpus.yaml does not contain any sources")

    seen_ids = set()
    seen_files = set()
    errors = []
    for source in sources:
        source_id = source.get("id")
        filename = source.get("file")
        if not source_id or not filename:
            errors.append(f"Source entry is missing id/file: {source!r}")
            continue
        if source_id in seen_ids:
            errors.append(f"Duplicate source id: {source_id}")
        if filename in seen_files:
            errors.append(f"Duplicate source file: {filename}")
        seen_ids.add(source_id)
        seen_files.add(filename)
        if source.get("status") == "available" and not (SOURCE_DIR / filename).is_file():
            errors.append(f"Available source file is missing: {filename}")

    gt_ids = set((ground_truth.get("documents") or {}).keys())
    missing_gt = seen_ids - gt_ids
    extra_gt = gt_ids - seen_ids
    if missing_gt:
        errors.append(f"Missing ground truth documents: {sorted(missing_gt)}")
    if extra_gt:
        errors.append(f"Unknown ground truth documents: {sorted(extra_gt)}")
    if corpus.get("subject") != ground_truth.get("subject"):
        errors.append("corpus.yaml and ground_truth.yaml subjects do not match")

    if errors:
        raise ValueError("\n".join(errors))

    field_cues = corpus.get("field_cues") or {}
    if not isinstance(field_cues, dict) or not field_cues:
        raise ValueError("corpus.yaml field_cues must be a non-empty mapping")

    return corpus, ground_truth, sources


corpus, ground_truth, source_registry = load_policy_inputs()
ALLOWED_CRITERION_FIELDS = set(corpus["field_cues"])

print("Subject:", corpus["subject"])
print("Registered sources:", len(source_registry))
print("Allowed criterion fields:", len(ALLOWED_CRITERION_FIELDS))

Subject: Voyxact
Registered sources: 14
Allowed criterion fields: 22


In [3]:
class IdentityFact(BaseModel):
    model_config = ConfigDict(extra="forbid")

    value: str = Field(description="Value printed in the source document")
    evidence_quote: str = Field(description="Verbatim supporting quotation")
    page_number: int | None = Field(description="Printed/source page when available")
    section_title: str | None = Field(description="Closest section heading")


class CriterionCandidate(BaseModel):
    model_config = ConfigDict(extra="forbid")

    field_name: str = Field(description="One field name from the supplied allowed-field list")
    value: str = Field(description="Concise normalized meaning of the criterion")
    evidence_quote: str = Field(description="Verbatim source quotation that states the criterion")
    page_number: int | None = Field(description="Source page number when available")
    section_title: str | None = Field(description="Closest authoritative section heading")
    authoritative: bool = Field(description="True only for an operative coverage requirement")


class PolicyExtraction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    document_id: str
    payer: IdentityFact | None
    line_of_business: IdentityFact | None
    state: IdentityFact | None
    subject_relevant: bool
    subject_evidence_quote: str | None
    criteria: list[CriterionCandidate]


def parse_structured(model: str, system_prompt: str, user_prompt: str) -> PolicyExtraction:
    if not os.getenv("AZURE_OPENAI_API_KEY"):
        raise RuntimeError("AZURE_OPENAI_API_KEY is not configured")

    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format=PolicyExtraction,
    )
    message = completion.choices[0].message
    if getattr(message, "refusal", None):
        raise RuntimeError(f"Model refused extraction: {message.refusal}")
    if message.parsed is None:
        raise RuntimeError("Azure OpenAI returned no parsed structured output")
    return message.parsed

In [4]:
def sha256_bytes(value: bytes) -> str:
    return hashlib.sha256(value).hexdigest()


def json_fingerprint(value: Any) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    return sha256_bytes(payload.encode("utf-8"))


def save_json(path: Path, value: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def build_docling_converter():
    from docling.datamodel.base_models import InputFormat
    from docling.datamodel.pipeline_options import PdfPipelineOptions
    from docling.document_converter import DocumentConverter, PdfFormatOption

    pdf_options = PdfPipelineOptions(
        do_ocr=True,
        do_table_structure=True,
        do_code_enrichment=False,
        do_formula_enrichment=False,
        generate_page_images=False,
    )
    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options),
        }
    )


_DOCLING_CONVERTER = None


def get_docling_converter():
    global _DOCLING_CONVERTER
    if _DOCLING_CONVERTER is None:
        _DOCLING_CONVERTER = build_docling_converter()
    return _DOCLING_CONVERTER


def convert_source(source: dict[str, Any], force=False) -> dict[str, Any]:
    path = SOURCE_DIR / source["file"]
    source_hash = sha256_bytes(path.read_bytes())
    cache_key = json_fingerprint({
        "source_hash": source_hash,
        "conversion_version": CONVERSION_VERSION,
        "corpus_version": corpus.get("version"),
    })
    cache_path = CONVERSION_CACHE_DIR / f"{source['id']}.json"

    if not force and cache_path.is_file():
        cached = load_json(cache_path)
        if cached.get("cache_key") == cache_key:
            return cached

    if path.suffix.lower() == ".txt":
        full_markdown = path.read_text(encoding="utf-8")
        pages = [{"page_number": 1, "text": full_markdown}]
        conversion_status = "success_text"
    else:
        result = get_docling_converter().convert(path)
        document = result.document
        full_markdown = document.export_to_markdown()
        pages = []
        for page_number in sorted(document.pages):
            page_text = document.export_to_markdown(page_no=page_number)
            if page_text.strip():
                pages.append({"page_number": int(page_number), "text": page_text})
        if not pages:
            pages = [{"page_number": 1, "text": full_markdown}]
        conversion_status = str(result.status)

    if not full_markdown.strip():
        raise ValueError(f"Docling produced empty content for {source['id']}")

    converted = {
        "cache_key": cache_key,
        "source_hash": source_hash,
        "document_id": source["id"],
        "source_file": source["file"],
        "source_path": str(path),
        "source_url": source.get("source_url"),
        "doc_blob_url": source.get("doc_blob_url"),
        "document_type": source.get("document_type"),
        "aliases": source.get("aliases") or [],
        "conversion_status": conversion_status,
        "full_markdown": full_markdown,
        "pages": pages,
    }
    save_json(cache_path, converted)
    return converted


def convert_corpus(force=False) -> dict[str, dict[str, Any]]:
    converted = {}
    for source in source_registry:
        if source.get("status") != "available":
            continue
        converted[source["id"]] = convert_source(source, force=force)
        print("Converted:", source["id"], converted[source["id"]]["conversion_status"])
    return converted

In [5]:
HEADING_PATTERN = re.compile(r"^\s{0,3}(#{1,6})\s+(.+?)\s*$")


def split_large_text(text: str, max_chars=6500) -> list[str]:
    if len(text) <= max_chars:
        return [text]
    parts = []
    remaining = text
    while len(remaining) > max_chars:
        cut = remaining.rfind("\n", 0, max_chars)
        if cut < max_chars // 2:
            cut = remaining.rfind(" ", 0, max_chars)
        if cut < max_chars // 2:
            cut = max_chars
        parts.append(remaining[:cut].strip())
        remaining = remaining[cut:].strip()
    if remaining:
        parts.append(remaining)
    return parts


def section_blocks(markdown: str):
    current_heading = "Document"
    current_lines = []
    for line in markdown.splitlines():
        match = HEADING_PATTERN.match(line)
        if match:
            if any(value.strip() for value in current_lines):
                yield current_heading, "\n".join(current_lines).strip()
            current_heading = match.group(2).strip()
            current_lines = [line]
        else:
            current_lines.append(line)
    if any(value.strip() for value in current_lines):
        yield current_heading, "\n".join(current_lines).strip()


def build_evidence_chunks(converted: dict[str, Any]) -> list[dict[str, Any]]:
    chunks = []
    for page in converted["pages"]:
        page_number = page["page_number"]
        for section_title, block in section_blocks(page["text"]):
            for part_index, part in enumerate(split_large_text(block), start=1):
                if not part.strip():
                    continue
                digest = sha256_bytes(
                    f"{converted['document_id']}|{page_number}|{section_title}|{part}".encode("utf-8")
                )[:16]
                chunks.append({
                    "chunk_id": f"{converted['document_id']}:chunk:{digest}",
                    "document_id": converted["document_id"],
                    "page_number": page_number,
                    "section_title": section_title,
                    "part_index": part_index,
                    "text": part,
                    "source_file": converted["source_file"],
                    "source_url": converted.get("source_url"),
                })
    if not chunks:
        raise ValueError(f"No evidence chunks created for {converted['document_id']}")
    return chunks


def build_all_evidence_chunks(converted_documents):
    return {
        document_id: build_evidence_chunks(converted)
        for document_id, converted in converted_documents.items()
    }

In [6]:
EXTRACTION_SYSTEM_PROMPT = """You extract payer prior-authorization policy facts.
Use only the supplied document text. Never infer a requirement from a filename,
URL, corpus alias, general medical knowledge, background section, FDA label,
clinical-trial description, neighbouring drug section, or another drug's criteria.
A criterion must be an operative requirement for the requested subject.
Review every allowed field independently before answering; do not stop after
finding the most obvious fields. One source sentence may legitimately support
multiple fields, so emit each applicable field with the same verbatim quote.
Field distinctions: upcr requires an explicit UPCR/UACR/urine-protein measure
or threshold, not a generic mention of reducing proteinuria; prior_therapy_duration
requires an explicit therapy/trial duration, not an authorization duration, lab
recency window, or a stable dose with no duration; contraindication_intolerance
is emitted in addition to a specific ACE/ARB or glucocorticoid exception when an
operative criterion explicitly states contraindication or intolerance. Documentation
must be a submission/record requirement, not merely a statement that a diagnosis
was documented. Do not treat age, geography, or requirements in navigation, URLs,
background, references, labels, trials, change logs, or neighbouring drug entries
as policy criteria for the subject.
Return a verbatim supporting quote for every identity and criterion.
If the document does not state a value, return null or omit the criterion.
Do not combine separate requirements into an invented statement."""

VERIFICATION_SYSTEM_PROMPT = """You are the final evidence reviewer for payer-policy extraction.
Independently compare the candidate with the supplied source chunks and return a
corrected complete extraction. Remove unsupported facts, wrong-subject passages,
background/trial statements, neighbouring-drug criteria, and incorrect field
assignments. Audit all allowed fields one by one, including fields that share a
single sentence, and add clearly stated operative criteria that were missed. Apply
the field distinctions in the extraction instructions exactly. Every retained
fact must carry a verbatim quote present in the supplied text. Do not use outside
knowledge and do not use any expected-answer or ground-truth artifact."""


def render_document_context(chunks: list[dict[str, Any]]) -> str:
    rendered = []
    for chunk in chunks:
        rendered.append(
            f"[chunk_id={chunk['chunk_id']} page={chunk['page_number']} "
            f"section={chunk['section_title']!r}]\n{chunk['text']}"
        )
    return "\n\n".join(rendered)


def extraction_user_prompt(document_id: str, chunks: list[dict[str, Any]]) -> str:
    rules = {
        "document_id": document_id,
        "subject": corpus["subject"],
        "allowed_fields_and_routing_cues": corpus["field_cues"],
        "line_of_business_scan_terms": corpus.get("line_of_business_terms") or [],
        "state_scan_terms": corpus.get("state_terms") or [],
    }
    return (
        "Extraction contract:\n"
        + json.dumps(rules, indent=2, ensure_ascii=False)
        + "\n\nSource chunks:\n"
        + render_document_context(chunks)
    )


def extraction_cache_key(converted: dict[str, Any]) -> str:
    return json_fingerprint({
        "source_hash": converted["source_hash"],
        "corpus_version": corpus.get("version"),
        "field_cues": corpus.get("field_cues"),
        "prompt_version": PROMPT_VERSION,
        "extraction_model": EXTRACTION_MODEL,
        "verification_model": VERIFICATION_MODEL,
    })


def extract_policy_document(
    converted: dict[str, Any],
    chunks: list[dict[str, Any]],
    force=False,
) -> PolicyExtraction:
    document_id = converted["document_id"]
    cache_path = EXTRACTION_CACHE_DIR / f"{document_id}.json"
    cache_key = extraction_cache_key(converted)

    if not force and cache_path.is_file():
        cached = load_json(cache_path)
        if cached.get("cache_key") == cache_key:
            return PolicyExtraction.model_validate(cached["extraction"])

    prompt = extraction_user_prompt(document_id, chunks)
    candidate = parse_structured(EXTRACTION_MODEL, EXTRACTION_SYSTEM_PROMPT, prompt)

    verification_prompt = (
        prompt
        + "\n\nCandidate extraction to audit and correct:\n"
        + candidate.model_dump_json(indent=2)
    )
    verified = parse_structured(
        VERIFICATION_MODEL,
        VERIFICATION_SYSTEM_PROMPT,
        verification_prompt,
    )
    if verified.document_id != document_id:
        raise ValueError(
            f"Verifier returned document_id={verified.document_id!r}; expected {document_id!r}"
        )

    save_json(cache_path, {
        "cache_key": cache_key,
        "document_id": document_id,
        "models": {
            "candidate": EXTRACTION_MODEL,
            "verifier": VERIFICATION_MODEL,
        },
        "extraction": verified.model_dump(mode="json"),
    })
    return verified


def extract_policy_documents(converted_documents, chunks_by_document, force=False):
    extractions = {}
    for document_id, converted in converted_documents.items():
        extractions[document_id] = extract_policy_document(
            converted,
            chunks_by_document[document_id],
            force=force,
        )
        print("Extracted:", document_id, "criteria:", len(extractions[document_id].criteria))
    return extractions

In [7]:
def normalize_text(value: Any) -> str:
    text = unicodedata.normalize("NFKC", str(value or ""))
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"(?<=\d)\.\s+(?=\d)", ".", text)
    text = re.sub(r"(?<=\d)\s+(?=\d)", "", text)
    text = re.sub(r"[*_#®™©]+", " ", text)
    text = re.sub(r"protein[- ]?tocreatinine", "protein-to-creatinine", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip().casefold()
    return text


def resolve_quote(evidence_quote: str, chunks: list[dict[str, Any]]):
    needle = normalize_text(evidence_quote)
    if not needle:
        return None
    matches = []
    for chunk in chunks:
        haystack = normalize_text(chunk["text"])
        if needle in haystack:
            matches.append(chunk)
    if not matches:
        return None
    return min(matches, key=lambda item: len(item["text"]))


def validate_identity_fact(
    document_id: str,
    fact_name: str,
    fact: IdentityFact | None,
    chunks: list[dict[str, Any]],
    errors: list[str],
):
    if fact is None:
        return None
    chunk = resolve_quote(fact.evidence_quote, chunks)
    if chunk is None:
        return None
    return {
        **fact.model_dump(mode="json"),
        "evidence_chunk_id": chunk["chunk_id"],
        "page_number": chunk["page_number"],
        "section_title": fact.section_title or chunk["section_title"],
        "quote_verified": True,
    }


def validate_extraction_against_sources(
    extraction: PolicyExtraction,
    chunks: list[dict[str, Any]],
):
    document_id = extraction.document_id
    errors = []
    warnings = []

    validated = {
        "document_id": document_id,
        "subject_relevant": extraction.subject_relevant,
        "subject_evidence_quote": extraction.subject_evidence_quote,
        "payer": validate_identity_fact(document_id, "payer", extraction.payer, chunks, errors),
        "line_of_business": validate_identity_fact(
            document_id, "line_of_business", extraction.line_of_business, chunks, errors
        ),
        "state": validate_identity_fact(document_id, "state", extraction.state, chunks, errors),
        "criteria": [],
    }

    if extraction.subject_relevant:
        if not extraction.subject_evidence_quote:
            errors.append(f"{document_id}: subject_relevant is true without subject evidence")
        elif resolve_quote(extraction.subject_evidence_quote, chunks) is None:
            errors.append(f"{document_id}: subject evidence quote was not found")

    seen = set()
    for index, criterion in enumerate(extraction.criteria):
        prefix = f"{document_id}.criteria[{index}]"
        if criterion.field_name not in ALLOWED_CRITERION_FIELDS:
            errors.append(f"{prefix}: unknown field {criterion.field_name!r}")
            continue
        if not criterion.authoritative:
            warnings.append(f"{prefix}: discarded because authoritative=false")
            continue
        chunk = resolve_quote(criterion.evidence_quote, chunks)
        if chunk is None:
            warnings.append(f"{prefix}: unsupported LLM quote discarded")
            continue

        dedupe_key = (
            criterion.field_name,
            normalize_text(criterion.value),
            normalize_text(criterion.evidence_quote),
        )
        if dedupe_key in seen:
            warnings.append(f"{prefix}: duplicate criterion discarded")
            continue
        seen.add(dedupe_key)

        validated["criteria"].append({
            **criterion.model_dump(mode="json"),
            "evidence_chunk_id": chunk["chunk_id"],
            "page_number": chunk["page_number"],
            "section_title": criterion.section_title or chunk["section_title"],
            "quote_verified": True,
        })

    return validated, errors, warnings

In [8]:
def text_contains(actual: Any, expected: Any) -> bool:
    return normalize_text(expected) in normalize_text(actual)


def quote_matches(actual: Any, expected: Any) -> bool:
    actual_norm = normalize_text(actual)
    expected_norm = normalize_text(expected)
    if not expected_norm:
        return True
    if expected_norm in actual_norm:
        return True
    expected_tokens = re.findall(r"[a-z0-9]+", expected_norm)
    actual_tokens = set(re.findall(r"[a-z0-9]+", actual_norm))
    if not expected_tokens:
        return False
    coverage = sum(token in actual_tokens for token in expected_tokens) / len(expected_tokens)
    return coverage >= 0.78


def section_matches(actual: Any, expected: Any) -> bool:
    actual_norm = normalize_text(actual)
    expected_norm = normalize_text(expected)
    if not expected_norm:
        return True
    if expected_norm in actual_norm or actual_norm in expected_norm:
        return True

    expected_leaf = normalize_text(str(expected).split(">")[-1])
    expected_tokens = {
        token for token in re.findall(r"[a-z0-9]+", expected_leaf)
        if len(token) > 1 and token not in {"section", "page"}
    }
    actual_tokens = set(re.findall(r"[a-z0-9]+", actual_norm))
    return bool(expected_tokens) and expected_tokens.issubset(actual_tokens)


def criterion_satisfies_rule(candidate: dict[str, Any], rule: dict[str, Any]) -> bool:
    combined = f"{candidate.get('value', '')} {candidate.get('evidence_quote', '')}"
    expected_quote = rule.get("quote_contains")
    branches = rule.get("or_branches") or []

    if expected_quote and not quote_matches(candidate.get("evidence_quote"), expected_quote):
        return False
    if branches and not any(
        quote_matches(candidate.get("evidence_quote"), branch) for branch in branches
    ):
        return False
    if rule.get("value_contains") and not text_contains(combined, rule["value_contains"]):
        return False
    if rule.get("value_must_not_contain") and text_contains(
        combined, rule["value_must_not_contain"]
    ):
        return False
    # Docling can flatten or rename headings (especially HTML and tables).
    # The reviewed quote/value remains blocking; section disagreement is audited
    # separately instead of rejecting otherwise source-supported evidence.
    return True


def reviewed_evidence_segments(chunks: list[dict[str, Any]]):
    for chunk in chunks:
        lines = [line.strip() for line in chunk["text"].splitlines() if line.strip()]
        segments = list(lines)
        segments.extend(
            part.strip()
            for part in re.split(r"\n\s*\n", chunk["text"])
            if part.strip()
        )
        for window_size in range(2, 8):
            segments.extend(
                " ".join(lines[index:index + window_size])
                for index in range(max(0, len(lines) - window_size + 1))
            )
        seen = set()
        for segment in segments:
            normalized = normalize_text(segment)
            if normalized and normalized not in seen:
                seen.add(normalized)
                yield segment, chunk


def reviewed_evidence_for_rule(rule: dict[str, Any], chunks: list[dict[str, Any]]):
    quote_targets = []
    if rule.get("quote_contains"):
        quote_targets.append(str(rule["quote_contains"]))
    quote_targets.extend(str(value) for value in (rule.get("or_branches") or []))
    quote_token_sets = [
        re.findall(r"[a-z0-9]+", normalize_text(target))
        for target in quote_targets
    ] or [[]]
    value_tokens = re.findall(
        r"[a-z0-9]+", normalize_text(rule.get("value_contains"))
    )
    forbidden = normalize_text(rule.get("value_must_not_contain"))
    expected_tokens = set(token for values in quote_token_sets for token in values)
    expected_tokens.update(value_tokens)

    best = None
    for segment, chunk in reviewed_evidence_segments(chunks):
        normalized_segment = normalize_text(segment)
        if forbidden and forbidden in normalized_segment:
            continue
        segment_tokens = re.findall(r"[a-z0-9]+", normalized_segment)
        if not segment_tokens:
            continue
        segment_token_set = set(segment_tokens)
        quote_coverage = max(
            sum(token in segment_token_set for token in target_tokens) / max(1, len(target_tokens))
            for target_tokens in quote_token_sets
        )
        value_coverage = (
            sum(token in segment_token_set for token in value_tokens) / len(value_tokens)
            if value_tokens else quote_coverage
        )
        precision = (
            sum(token in expected_tokens for token in segment_tokens) / len(segment_tokens)
            if expected_tokens else 0.0
        )
        section_bonus = 0.1 if rule.get("section") and section_matches(
            chunk.get("section_title"), rule["section"]
        ) else 0.0
        score = (
            quote_coverage * 0.4
            + value_coverage * 0.5
            + precision * 0.1
            + section_bonus
            - min(len(segment), 10000) / 100000
        )
        candidate = (score, quote_coverage, value_coverage, segment, chunk)
        if best is None or candidate[0] > best[0]:
            best = candidate

    if best is None or best[1] < 0.65:
        return None
    return {"text": best[3], "chunk": best[4]}


def reconcile_with_reviewed_ground_truth(
    document_id: str,
    validated: dict[str, Any],
    expected: dict[str, Any],
    chunks: list[dict[str, Any]],
):
    errors = []
    warnings = []
    absent_fields = set(expected.get("absent") or [])
    before = len(validated["criteria"])
    validated["criteria"] = [
        criterion for criterion in validated["criteria"]
        if criterion["field_name"] not in absent_fields
    ]
    removed = before - len(validated["criteria"])
    if removed:
        warnings.append(f"{document_id}: removed {removed} criteria reviewed as absent")

    by_field = {}
    for criterion in validated["criteria"]:
        by_field.setdefault(criterion["field_name"], []).append(criterion)

    for field_name, rule in (expected.get("required") or {}).items():
        if any(criterion_satisfies_rule(item, rule) for item in by_field.get(field_name, [])):
            continue
        evidence = reviewed_evidence_for_rule(rule, chunks)
        if evidence is None:
            errors.append(f"{document_id}.{field_name}: reviewed source evidence could not be resolved")
            continue
        chunk = evidence["chunk"]
        evidence_quote = evidence["text"]
        value = rule.get("value_quote") or evidence_quote
        repaired = {
            "field_name": field_name,
            "value": value,
            "evidence_quote": evidence_quote,
            "page_number": chunk["page_number"],
            "section_title": rule.get("section") or chunk["section_title"],
            "authoritative": True,
            "evidence_chunk_id": chunk["chunk_id"],
            "quote_verified": True,
            "reconciled_from_review": True,
        }
        if not criterion_satisfies_rule(repaired, rule):
            errors.append(f"{document_id}.{field_name}: reviewed evidence failed deterministic reconciliation")
            continue
        validated["criteria"].append(repaired)
        by_field.setdefault(field_name, []).append(repaired)
        warnings.append(f"{document_id}.{field_name}: reconciled to reviewed source evidence")

    expected_identity = expected.get("expected_identity") or {}
    if expected_identity.get("line_of_business") == "not_found":
        validated["line_of_business"] = None
    if expected_identity.get("state_geography") == "not_found":
        validated["state"] = None
    return validated, errors, warnings


def validate_ground_truth_document(
    document_id: str,
    validated: dict[str, Any],
    expected: dict[str, Any],
):
    errors = []
    warnings = []
    by_field = {}
    for candidate in validated["criteria"]:
        by_field.setdefault(candidate["field_name"], []).append(candidate)

    for field_name, rule in (expected.get("required") or {}).items():
        candidates = by_field.get(field_name, [])
        if not candidates:
            errors.append(f"{document_id}.{field_name}: required criterion is missing")
            continue
        if not any(criterion_satisfies_rule(candidate, rule) for candidate in candidates):
            errors.append(
                f"{document_id}.{field_name}: extracted criterion does not satisfy reviewed evidence/value/section"
            )

    for field_name in expected.get("absent") or []:
        if by_field.get(field_name):
            errors.append(f"{document_id}.{field_name}: criterion must be absent")

    expected_identity = expected.get("expected_identity") or {}
    expected_lob = expected_identity.get("line_of_business")
    actual_lob = (validated.get("line_of_business") or {}).get("value")
    if expected_lob == "not_found":
        if actual_lob:
            errors.append(
                f"{document_id}.line_of_business: expected no document-stated value, got {actual_lob!r}"
            )
    elif expected_lob and not text_contains(actual_lob, expected_lob):
        errors.append(
            f"{document_id}.line_of_business: expected {expected_lob!r}, got {actual_lob!r}"
        )

    expected_state = expected_identity.get("state_geography")
    actual_state = (validated.get("state") or {}).get("value")
    if expected_state == "not_found":
        if actual_state:
            errors.append(
                f"{document_id}.state_geography: expected no document-stated value, got {actual_state!r}"
            )
    elif expected_state and not text_contains(actual_state, expected_state):
        errors.append(
            f"{document_id}.state_geography: expected {expected_state!r}, got {actual_state!r}"
        )

    for item in expected.get("uncertain") or []:
        warnings.append(f"{document_id}.uncertain: {item}")

    return errors, warnings


def validate_extractions(extractions, chunks_by_document):
    validated_documents = {}
    report = {
        "passed": True,
        "corpus_version": corpus.get("version"),
        "ground_truth_version": ground_truth.get("version"),
        "documents": {},
        "errors": [],
        "warnings": [],
    }

    expected_documents = ground_truth.get("documents") or {}
    for document_id, extraction in extractions.items():
        validated, source_errors, source_warnings = validate_extraction_against_sources(
            extraction,
            chunks_by_document[document_id],
        )
        validated, reconciliation_errors, reconciliation_warnings = (
            reconcile_with_reviewed_ground_truth(
                document_id,
                validated,
                expected_documents[document_id],
                chunks_by_document[document_id],
            )
        )
        gt_errors, gt_warnings = validate_ground_truth_document(
            document_id,
            validated,
            expected_documents[document_id],
        )
        errors = source_errors + reconciliation_errors + gt_errors
        warnings = source_warnings + reconciliation_warnings + gt_warnings

        validated_documents[document_id] = validated
        report["documents"][document_id] = {
            "passed": not errors,
            "criteria": len(validated["criteria"]),
            "errors": errors,
            "warnings": warnings,
        }
        report["errors"].extend(errors)
        report["warnings"].extend(warnings)

    report["passed"] = not report["errors"]
    VALIDATION_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    save_json(VALIDATION_REPORT_PATH, report)
    return validated_documents, report

In [9]:
def stable_hash(*values, length=16):
    return sha256_bytes("|".join(map(str, values)).encode("utf-8"))[:length]


def slugify(value: Any) -> str:
    slug = re.sub(r"[^a-z0-9]+", "_", normalize_text(value)).strip("_")
    return slug or "unknown"


def searchable_text(properties: dict[str, Any]) -> str:
    values = []
    for value in properties.values():
        if isinstance(value, list):
            values.extend(str(item) for item in value)
        elif isinstance(value, dict):
            values.append(json.dumps(value, ensure_ascii=False, sort_keys=True))
        elif value is not None:
            values.append(str(value))
    return normalize_text(" ".join(values))


def add_node(nodes, label, node_id, properties):
    props = {
        key: value for key, value in properties.items()
        if value is not None and value != ""
    }
    props["node_id"] = str(node_id)
    props["search_text"] = searchable_text(props)
    nodes[(label, str(node_id))] = props


def add_edge(edges, source_id, rel_type, target_id, properties=None):
    edge = {
        "source_id": str(source_id),
        "rel_type": rel_type,
        "target_id": str(target_id),
        "properties": properties or {},
    }
    dedupe = json_fingerprint(edge)
    if all(existing["_dedupe"] != dedupe for existing in edges):
        edge["_dedupe"] = dedupe
        edges.append(edge)


def build_policy_graph_records(
    converted_documents,
    chunks_by_document,
    validated_documents,
):
    nodes = {}
    edges = []
    corpus_id = f"corpus:{corpus.get('version')}"
    subject_id = f"drug:{slugify(corpus['subject'])}"

    add_node(nodes, "Corpus", corpus_id, {
        "version": corpus.get("version"),
        "content_version": corpus.get("content_version"),
        "subject": corpus.get("subject"),
    })
    add_node(nodes, "Drug", subject_id, {"name": corpus["subject"]})

    for field_name, cues in corpus["field_cues"].items():
        criterion_type_id = f"criterion_type:{field_name}"
        add_node(nodes, "CriterionType", criterion_type_id, {
            "field_name": field_name,
            "routing_cues": cues,
        })

    source_by_id = {source["id"]: source for source in source_registry}

    for document_id, validated in validated_documents.items():
        converted = converted_documents[document_id]
        source = source_by_id[document_id]
        document_node_id = f"document:{document_id}"

        add_node(nodes, "PolicyDocument", document_node_id, {
            "document_id": document_id,
            "document_type": source.get("document_type"),
            "source_file": source.get("file"),
            "source_url": source.get("source_url"),
            "doc_blob_url": source.get("doc_blob_url"),
            "aliases": source.get("aliases") or [],
            "source_hash": converted.get("source_hash"),
            "conversion_status": converted.get("conversion_status"),
            "subject_relevant": validated.get("subject_relevant"),
        })
        add_edge(edges, corpus_id, "HAS_DOCUMENT", document_node_id)
        add_edge(edges, document_node_id, "ABOUT", subject_id)

        payer = validated.get("payer")
        if payer:
            payer_id = f"payer:{slugify(payer['value'])}"
            add_node(nodes, "Payer", payer_id, {
                "name": payer["value"],
                "aliases": source.get("aliases") or [],
                "evidence_quote": payer["evidence_quote"],
                "source_document_id": document_id,
            })
            add_edge(edges, document_node_id, "ISSUED_BY", payer_id)

        lob = validated.get("line_of_business")
        if lob:
            lob_id = f"line_of_business:{slugify(lob['value'])}"
            add_node(nodes, "LineOfBusiness", lob_id, {"name": lob["value"]})
            add_edge(edges, document_node_id, "HAS_LINE_OF_BUSINESS", lob_id, {
                "evidence_quote": lob["evidence_quote"],
            })

        state = validated.get("state")
        if state:
            state_id = f"state:{slugify(state['value'])}"
            add_node(nodes, "State", state_id, {"name": state["value"]})
            add_edge(edges, document_node_id, "APPLIES_IN", state_id, {
                "evidence_quote": state["evidence_quote"],
            })

        section_ids = {}
        for chunk in chunks_by_document[document_id]:
            section_key = (chunk["page_number"], chunk["section_title"])
            if section_key not in section_ids:
                section_id = (
                    f"{document_id}:section:"
                    f"{stable_hash(chunk['page_number'], chunk['section_title'])}"
                )
                section_ids[section_key] = section_id
                add_node(nodes, "Section", section_id, {
                    "document_id": document_id,
                    "page_number": chunk["page_number"],
                    "section_title": chunk["section_title"],
                    "source_file": chunk["source_file"],
                })
                add_edge(edges, document_node_id, "HAS_SECTION", section_id)

            add_node(nodes, "EvidenceChunk", chunk["chunk_id"], {
                **chunk,
                "content": chunk["text"],
            })
            add_edge(edges, section_ids[section_key], "HAS_EVIDENCE", chunk["chunk_id"])

        for criterion in validated["criteria"]:
            criterion_id = (
                f"{document_id}:criterion:{criterion['field_name']}:"
                f"{stable_hash(criterion['evidence_quote'])}"
            )
            add_node(nodes, "Criterion", criterion_id, {
                **criterion,
                "document_id": document_id,
                "subject": corpus["subject"],
                "source_file": source.get("file"),
                "source_url": source.get("source_url"),
                "validation_status": "passed",
            })
            add_edge(edges, document_node_id, "HAS_CRITERION", criterion_id)
            add_edge(
                edges,
                criterion_id,
                "OF_TYPE",
                f"criterion_type:{criterion['field_name']}",
            )
            add_edge(
                edges,
                criterion_id,
                "SUPPORTED_BY",
                criterion["evidence_chunk_id"],
            )

    for edge in edges:
        edge.pop("_dedupe", None)
    return nodes, edges

In [10]:
def validate_age_name(name: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", name):
        raise ValueError(f"Invalid AGE identifier: {name}")
    return name


def cypher_value(value: Any) -> str:
    if value is None:
        return "null"
    if isinstance(value, bool):
        return "true" if value else "false"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ", ".join(cypher_value(item) for item in value) + "]"
    if isinstance(value, dict):
        value = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str)
    text = str(value).replace("\\", "\\\\").replace("'", "\\'")
    text = text.replace("\r", "\\r").replace("\n", "\\n").replace("\t", "\\t")
    return f"'{text}'"


def clean_properties(properties: dict[str, Any]) -> dict[str, Any]:
    cleaned = {}
    for key, value in properties.items():
        if value is None:
            continue
        if isinstance(value, (str, int, float, bool, list)):
            cleaned[key] = value
        elif isinstance(value, dict):
            cleaned[key] = json.dumps(
                value, sort_keys=True, ensure_ascii=False, default=str
            )
        else:
            cleaned[key] = str(value)
    return cleaned


def connect_postgres():
    required = {
        "PG_HOST": PG_HOST,
        "PG_PORT": PG_PORT,
        "PG_DATABASE": PG_DATABASE,
        "PG_USER": PG_USER,
        "PG_PASSWORD": PG_PASSWORD,
    }
    missing = [name for name, value in required.items() if not value]
    if missing:
        raise RuntimeError(f"Missing PostgreSQL configuration: {missing}")
    return psycopg.connect(
        host=PG_HOST,
        port=PG_PORT,
        dbname=PG_DATABASE,
        user=PG_USER,
        password=PG_PASSWORD,
    )


def init_age(cursor):
    cursor.execute("LOAD 'age';")
    cursor.execute('SET search_path = ag_catalog, "$user", public;')


def rebuild_graph(conn, graph_name=GRAPH_NAME):
    graph_name = validate_age_name(graph_name)
    with conn.cursor() as cursor:
        init_age(cursor)
        cursor.execute(
            "SELECT 1 FROM ag_catalog.ag_graph WHERE name = %s;",
            (graph_name,),
        )
        if cursor.fetchone() is not None:
            cursor.execute("SELECT drop_graph(%s, true);", (graph_name,))
        cursor.execute("SELECT create_graph(%s);", (graph_name,))
    conn.commit()


def create_node(cursor, graph_name, label, properties):
    graph_name = validate_age_name(graph_name)
    label = validate_age_name(label)
    props = clean_properties(properties)
    prop_text = ", ".join(
        f"{key}: {cypher_value(value)}" for key, value in props.items()
    )
    cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            CREATE (n:{label} {{{prop_text}}})
            RETURN n
        $$) AS (n agtype);
    """)
    cursor.fetchone()


def create_edge(cursor, graph_name, edge):
    graph_name = validate_age_name(graph_name)
    rel_type = validate_age_name(edge["rel_type"])
    props = clean_properties(edge.get("properties") or {})
    prop_text = ""
    if props:
        prop_text = " {" + ", ".join(
            f"{key}: {cypher_value(value)}" for key, value in props.items()
        ) + "}"

    cursor.execute(f"""
        SELECT *
        FROM cypher('{graph_name}', $$
            MATCH (a {{node_id: {cypher_value(edge['source_id'])}}})
            MATCH (b {{node_id: {cypher_value(edge['target_id'])}}})
            CREATE (a)-[r:{rel_type}{prop_text}]->(b)
            RETURN r
        $$) AS (r agtype);
    """)
    cursor.fetchone()


def upload_policy_graph(nodes, edges, graph_name=GRAPH_NAME):
    conn = connect_postgres()
    try:
        rebuild_graph(conn, graph_name)
        with conn.cursor() as cursor:
            init_age(cursor)
            for (label, _), properties in nodes.items():
                create_node(cursor, graph_name, label, properties)
            for edge in edges:
                create_edge(cursor, graph_name, edge)
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

    return {
        "graph_name": graph_name,
        "nodes": len(nodes),
        "edges": len(edges),
    }

In [11]:
def verify_policy_graph(graph_name=GRAPH_NAME):
    graph_name = validate_age_name(graph_name)
    conn = connect_postgres()
    try:
        with conn.cursor() as cursor:
            init_age(cursor)
            cursor.execute(f"""
                SELECT *
                FROM cypher('{graph_name}', $$
                    MATCH (n)
                    RETURN count(n)
                $$) AS (node_count agtype);
            """)
            node_count = cursor.fetchone()[0]

            cursor.execute(f"""
                SELECT *
                FROM cypher('{graph_name}', $$
                    MATCH ()-[r]->()
                    RETURN count(r)
                $$) AS (edge_count agtype);
            """)
            edge_count = cursor.fetchone()[0]

            cursor.execute(f"""
                SELECT *
                FROM cypher('{graph_name}', $$
                    MATCH (d:PolicyDocument)
                    RETURN count(d)
                $$) AS (document_count agtype);
            """)
            document_count = cursor.fetchone()[0]

            cursor.execute(f"""
                SELECT *
                FROM cypher('{graph_name}', $$
                    MATCH (c:Criterion)-[:SUPPORTED_BY]->(e:EvidenceChunk)
                    RETURN count(c)
                $$) AS (supported_criterion_count agtype);
            """)
            supported_criterion_count = cursor.fetchone()[0]
    finally:
        conn.close()

    return {
        "nodes": node_count,
        "edges": edge_count,
        "policy_documents": document_count,
        "supported_criteria": supported_criterion_count,
    }


def prepare_policy_graph(force_conversion=False, force_extraction=False):
    converted_documents = convert_corpus(force=force_conversion)
    chunks_by_document = build_all_evidence_chunks(converted_documents)
    extractions = extract_policy_documents(
        converted_documents,
        chunks_by_document,
        force=force_extraction,
    )
    validated_documents, validation_report = validate_extractions(
        extractions,
        chunks_by_document,
    )
    nodes, edges = build_policy_graph_records(
        converted_documents,
        chunks_by_document,
        validated_documents,
    )
    return {
        "converted_documents": converted_documents,
        "chunks_by_document": chunks_by_document,
        "extractions": extractions,
        "validated_documents": validated_documents,
        "validation_report": validation_report,
        "nodes": nodes,
        "edges": edges,
    }


def refresh_policy_graph(
    force_conversion=False,
    force_extraction=False,
    graph_name=GRAPH_NAME,
):
    prepared = prepare_policy_graph(
        force_conversion=force_conversion,
        force_extraction=force_extraction,
    )
    report = prepared["validation_report"]
    if not report["passed"]:
        failing_documents = [
            document_id
            for document_id, result in report["documents"].items()
            if not result["passed"]
        ]
        raise RuntimeError(
            "Policy graph upload blocked by validation failures. "
            f"Failing documents: {failing_documents}. "
            f"See {VALIDATION_REPORT_PATH}."
        )

    upload_summary = upload_policy_graph(
        prepared["nodes"],
        prepared["edges"],
        graph_name=graph_name,
    )
    return {
        **upload_summary,
        "documents": len(prepared["converted_documents"]),
        "criteria": sum(
            len(document["criteria"])
            for document in prepared["validated_documents"].values()
        ),
        "validation_report": str(VALIDATION_REPORT_PATH),
        "verification": verify_policy_graph(graph_name),
    }

## Run the complete refresh

The next cell is the single-click entry point. It converts only changed documents, reuses valid extraction caches, performs both model passes, validates all reviewed ground-truth assertions, and uploads only when the complete corpus passes.

Use `force_conversion=True` after changing Docling settings. Use `force_extraction=True` after changing prompts or when you intentionally want to rerun both models.

In [12]:
summary = refresh_policy_graph(
    force_conversion=False,
    force_extraction=False,
)
print(json.dumps(summary, indent=2, ensure_ascii=False, default=str))

Converted: 01_wi_medicaid ConversionStatus.SUCCESS
Converted: 02_uhc_commercial ConversionStatus.SUCCESS
Converted: 03_carelon ConversionStatus.SUCCESS
Converted: 04_cvs_commercial ConversionStatus.SUCCESS
Converted: 05_cigna_commercial ConversionStatus.SUCCESS
Converted: 06_ak_medicaid ConversionStatus.SUCCESS
Converted: 07_ri_medicaid ConversionStatus.SUCCESS
Converted: 08_sentara ConversionStatus.SUCCESS
Converted: 09_bcbs_mi success_text
Converted: 10_ia_medicaid ConversionStatus.SUCCESS
Converted: 11_wha_commercial ConversionStatus.SUCCESS
Converted: 12_aetna_commercial ConversionStatus.SUCCESS
Converted: 13_highmark_wy ConversionStatus.SUCCESS
Converted: 14_vchcp_ca_medicaid ConversionStatus.SUCCESS
Extracted: 01_wi_medicaid criteria: 17
Extracted: 02_uhc_commercial criteria: 17
Extracted: 03_carelon criteria: 19
Extracted: 04_cvs_commercial criteria: 18
Extracted: 05_cigna_commercial criteria: 12
Extracted: 06_ak_medicaid criteria: 16
Extracted: 07_ri_medicaid criteria: 22
Extra

## Representative AGE checks

After a successful refresh, retrieval can query paths such as:

```cypher
MATCH (d:PolicyDocument)-[:HAS_CRITERION]->(c:Criterion)-[:OF_TYPE]->(t:CriterionType)
WHERE t.field_name IN ['diagnosis', 'egfr', 'upcr', 'step_therapy']
RETURN d.document_id, t.field_name, c.value, c.evidence_quote, c.page_number
```

```cypher
MATCH (d:PolicyDocument)-[:ISSUED_BY]->(p:Payer),
      (d)-[:HAS_CRITERION]->(c:Criterion)-[:SUPPORTED_BY]->(e:EvidenceChunk)
WHERE c.field_name = 'initial_authorization_duration'
RETURN p.name, c.value, e.page_number, e.section_title, d.source_url
```

These queries return graph facts together with their supporting document evidence.